In [12]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from openpyxl import Workbook
import time
import os

# 테스트용 설정
high_cnt = 100
low_cnt = 100

# 엑셀에서 URL만 읽어오기
df = pd.read_excel(r'C:\py_temp\미국_ulta_상품정보\미국_ulta_상품정보.xlsx')
urls = df.iloc[:, 8].tolist()

# 저장 파일명 설정
n = time.localtime()
s = '%04d-%02d-%02d-%02d-%02d-%02d' % (n.tm_year, n.tm_mon, n.tm_mday, n.tm_hour, n.tm_min, n.tm_sec)

fx_review_name = fr'C:\py_temp\미국_ulta_상품정보\{s}_ulta_reviews.xlsx'
fc_review_name = fr'C:\py_temp\미국_ulta_상품정보\{s}_ulta_reviews.csv'

# 드라이버 실행
driver = webdriver.Chrome()

# 엑셀 시트 생성
wb = Workbook()
wb.remove(wb.active)
for i in range(1, len(urls) + 1):
    ws = wb.create_sheet(title=f"Sheet{i}")
    ws.append(['Review_ID', 'Rating', 'Review_Text', 'Sorter'])

wb.save(fx_review_name)
print(f"총 {len(urls)}개 상품 / 시트 {len(urls)}개 생성 완료!")
print(f"엑셀: {fx_review_name}")
print(f"CSV: {fc_review_name}")

총 10개 상품 / 시트 10개 생성 완료!
엑셀: C:\py_temp\미국_ulta_상품정보\2026-04-27-11-50-01_ulta_reviews.xlsx
CSV: C:\py_temp\미국_ulta_상품정보\2026-04-27-11-50-01_ulta_reviews.csv


In [13]:
total_count = 0

for sheet_idx, url in enumerate(urls, 1):
    print(f"\n{'='*50}")
    print(f"[{sheet_idx}/{len(urls)}] 시작: {url}")

    review_id = 1
    
    driver.get(url)
    driver.maximize_window()
    time.sleep(3)
    
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)
    print("  스크롤 완료")

    ws = wb[f"Sheet{sheet_idx}"]
    
    for sort_type in ['Highest Rated', 'Lowest Rated']:
        from selenium.webdriver.support.ui import Select
        try:
            select = Select(driver.find_element(By.ID, 'pr-rd-sort-by'))
            select.select_by_visible_text(sort_type)
            time.sleep(2)
        except Exception as e:
            print(f"  정렬 실패: {e}")
            continue
        
        sorter_label = '높은 순' if sort_type == 'Highest Rated' else '낮은 순'
        cnt = high_cnt if sort_type == 'Highest Rated' else low_cnt
        collected = 0
        
        print(f"\n  [{sorter_label}] 수집 시작 (목표: {cnt}개)")
        
        while collected < cnt:
            reviews = driver.find_elements(By.XPATH, '//div[@class="pr-review"]')
            print(f"  현재 페이지 리뷰 수: {len(reviews)}개")
            
            for review in reviews:
                if collected >= cnt:
                    break
                
                # 별점
                try:
                    aria = review.find_element(By.XPATH, './/div[contains(@class, "pr-snippet-stars")]').get_attribute('aria-label')
                    rating = aria.split('Rated ')[1].split(' out')[0] + '.0'
                except:
                    rating = ''
                
                # 리뷰 제목
                try:
                    title = review.find_element(By.XPATH, './/h5[contains(@class, "pr-rd-review-headline")]').text.strip()
                except:
                    title = ''
                
                # 본문
                try:
                    body = review.find_element(By.XPATH, './/p[contains(@class, "pr-rd-description-text")]').text.strip()
                except:
                    body = ''
                
                review_text = f"{title}\n{body}"
                collected += 1
                
                ws.append([review_id, rating, review_text, f'{sorter_label} {collected}번째'])
                review_id += 1
                
                print(f"    [{sorter_label} {collected}번째] Rating: {rating} | Review_Text: {title[:20]}...")
            
            # 다음 페이지
            if collected < cnt:
                try:
                    next_btn = driver.find_element(By.XPATH, '//a[contains(@class, "pr-rd-pagination-btn--next")]')
                    driver.execute_script("arguments[0].click();", next_btn)
                    time.sleep(2)
                    print(f"  다음 페이지로 이동")
                except:
                    print(f"  다음 페이지 없음 - 수집 종료")
                    break
        
        print(f"  [{sorter_label}] 수집 완료: {collected}개")
        total_count += collected
wb.save(fx_review_name)
wb.save(fx_review_name)

# CSV 저장
all_data = []
for sheet_idx in range(1, len(urls) + 1):
    ws = wb[f"Sheet{sheet_idx}"]
    for row in ws.iter_rows(min_row=2, values_only=True):  # 헤더 제외
        all_data.append(row)

csv_df = pd.DataFrame(all_data, columns=['Review_ID', 'Rating', 'Review_Text', 'Sorter'])
csv_df.to_csv(fc_review_name, encoding='utf-8-sig', index=False)

print(f"\n{'='*50}")
print(f"전체 완료! 총 {total_count}개 리뷰 수집")
print(f"엑셀: {fx_review_name}")
print(f"CSV: {fc_review_name}")


[1/10] 시작: https://www.ulta.com/p/dewy-skin-cream-line-plumping-moisturizer-pimprod2049087?sku=2634098
  스크롤 완료

  [높은 순] 수집 시작 (목표: 100개)
  현재 페이지 리뷰 수: 5개
    [높은 순 1번째] Rating: 5.0 | Review_Text: Dewy glowy...
    [높은 순 2번째] Rating: 5.0 | Review_Text: would buy again...
    [높은 순 3번째] Rating: 5.0 | Review_Text: My go to for dry ski...
    [높은 순 4번째] Rating: 5.0 | Review_Text: Hope to continue buy...
    [높은 순 5번째] Rating: 5.0 | Review_Text: Love my Tatcha!...
  다음 페이지로 이동
  현재 페이지 리뷰 수: 5개
    [높은 순 6번째] Rating: 5.0 | Review_Text: Great for Tret Users...
    [높은 순 7번째] Rating: 5.0 | Review_Text: Love it!...
    [높은 순 8번째] Rating: 5.0 | Review_Text: Worth Every Penny...
    [높은 순 9번째] Rating: 5.0 | Review_Text: Love this cream!...
    [높은 순 10번째] Rating: 5.0 | Review_Text: Amazing...
  다음 페이지로 이동
  현재 페이지 리뷰 수: 5개
    [높은 순 11번째] Rating: 5.0 | Review_Text: Nothing compares...
    [높은 순 12번째] Rating: 5.0 | Review_Text: Love it!...
    [높은 순 13번째] Rating: 5.0 | Review_Text: I would bu